# 02 - Distributed Training with Kubeflow & Feast

![Training Workflow](../../docs/diagrams/02-training-workflow.png)

## What This Notebook Does

| Step | Component | Action |
|------|-----------|--------|
| 1 | Kubeflow SDK | Submit TrainJob to cluster |
| 2 | Feast Remote Client | Feature retrieval via gRPC (operator-managed) |
| 3 | PyTorch DDP | Multi-GPU distributed training |
| 4 | MLflow | Track experiments, log model |

## Architecture

| Component | Service | Purpose |
|-----------|---------|---------|
| **Kubeflow Trainer** | TrainJob CRD | Orchestrates distributed training pods |
| **Feast Operator** | gRPC services | Provides remote client config for feature retrieval |
| **MLflow Operator** | RHOAI MLflow | Experiment tracking with workspace isolation |
| **PyTorch DDP** | Training pods | Distributed model training across GPUs |

**Prerequisites:**
- Features registered and materialized in Feast (run `01a-local.ipynb` or `01b-remote.ipynb`)
- MLflow Operator deployed (see `manifests/mlflow/`)
- Feast Operator deployed with FeatureStore CR

In [ ]:
# Install required packages (restart kernel after running)
# Kubeflow SDK from Red Hat AI index, MLflow from Red Hat fork for workspace support
%pip install kubeflow --no-cache-dir --index-url https://console.redhat.com/api/pypi/public-rhai/rhoai/3.3/cuda12.9-ubi9/simple/
%pip install -q kubernetes "git+https://github.com/red-hat-data-services/mlflow@master"

# Note: torch, numpy, pandas are expected to be in the workbench image

In [ ]:
%pip show kubeflow

## Configuration

| Parameter | Value | Purpose |
|-----------|-------|----------|
| `NAMESPACE` | `feast-trainer-demo` | K8s namespace |
| `PVC` | `shared` | Persistent storage for model artifacts |
| `RUNTIME` | `torch-distributed` | ClusterTrainingRuntime for DDP |
| `EPOCHS` | 50 | Training iterations |
| `FEAST_CONFIG_PATH` | `/opt/app-root/src/feast-config/salesforecasting` | Feast remote client config (operator-provided) |
| `MLFLOW_URI` | Internal service | RHOAI MLflow tracking service |
| `MLFLOW_WORKSPACE` | `feast-trainer-demo` | MLflow workspace (namespace isolation) |

In [ ]:
NAMESPACE = "feast-trainer-demo"
PVC = "shared"
RUNTIME = "torch-distributed"
EPOCHS = 50

# Feast remote client config (mounted by OpenShift AI workbench from Feast Operator)
FEAST_CONFIG_PATH = "/opt/app-root/src/feast-config/salesforecasting"

# RHOAI MLflow Configuration (internal service - pods access via cluster network)
# Note: For browser access, use the OAuth proxy route instead
CLUSTER_DOMAIN = "apps.oai-kft-ibm.ibm.rh-ods.com"
MLFLOW_INTERNAL_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443/mlflow"
MLFLOW_EXTERNAL_URI = f"https://mlflow-ui-redhat-ods-applications.{CLUSTER_DOMAIN}"
MLFLOW_WORKSPACE = NAMESPACE  # Workspace = namespace for isolation

## Authentication

Connect to Kubernetes and RHOAI MLflow using OAuth token:

| Token | Purpose |
|-------|---------|
| `K8S_TOKEN` | Kubernetes API access |
| `MLFLOW_TOKEN` | RHOAI MLflow tracking (same as OCP token) |

In [ ]:
import os
import subprocess

# Try to get token dynamically from 'oc whoami --show-token'
# Fallback to K8S_TOKEN env var or in-cluster service account
try:
    K8S_TOKEN = subprocess.check_output(['oc', 'whoami', '--show-token'], text=True).strip()
except:
    K8S_TOKEN = os.getenv("K8S_TOKEN", "")
    if not K8S_TOKEN:
        # Try in-cluster service account token
        token_path = '/var/run/secrets/kubernetes.io/serviceaccount/token'
        if os.path.exists(token_path):
            with open(token_path) as f:
                K8S_TOKEN = f.read().strip()

K8S_API = os.getenv("K8S_API_SERVER", f"https://api.{CLUSTER_DOMAIN.replace('apps.', '')}:6443")
MLFLOW_TOKEN = K8S_TOKEN  # Same token for RHOAI MLflow
    
print(f"K8S API: {K8S_API}")
print(f"Token: {K8S_TOKEN[:20]}..." if K8S_TOKEN else "No token found (will use in-cluster auth)")

## Initialize Kubeflow TrainerClient

The `TrainerClient` manages TrainJob lifecycle:
- `train()` → Submit job
- `get_job()` → Check status
- `wait_for_job_status()` → Block until complete
- `delete_job()` → Cleanup

In [ ]:
from kubernetes import client as k8s
from kubeflow.trainer import TrainerClient, CustomTrainer
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer.options import Labels, PodTemplateOverrides, PodTemplateOverride, PodSpecOverride, ContainerOverride

cfg = k8s.Configuration()
if K8S_TOKEN:
    cfg.host = K8S_API
    cfg.verify_ssl = False
    cfg.api_key = {"authorization": f"Bearer {K8S_TOKEN}"}

trainer = TrainerClient(KubernetesBackendConfig(namespace=NAMESPACE, client_configuration=cfg if K8S_TOKEN else None))
runtime = trainer.get_runtime(RUNTIME)
print(f"Runtime: {RUNTIME}")

## Training Function

The training function is defined in `training_script.py` (same directory).

### Key Components:

| Step | Component | Description |
|------|-----------|-------------|
| 1 | **Feast Remote Client** | Retrieves features via gRPC (operator-managed) |
| 2 | **PyTorch DDP** | Distributed training across GPUs |
| 3 | **MLflow Registry** | Logs model and registers in Model Registry |

### Function Signature:
```python
def train_fn(epochs, namespace, output_dir, feast_config_path):
    # 1. Feast feature retrieval (rank 0)
    # 2. DDP training (all ranks)
    # 3. MLflow logging & model registry (rank 0)
```

In [ ]:
# Load training function from training_script.py (same directory as this notebook)
from training_script import train_fn

print("✅ train_fn loaded")

## Submit TrainJob

The `CustomTrainer` packages the function for distributed execution:

| Parameter | Value | Purpose |
|-----------|-------|----------|
| `func` | `train_fn` | Python function to run |
| `func_args` | `{epochs, namespace, output_dir, feast_config_path}` | Arguments passed as kwargs |
| `num_nodes` | 1 | Number of worker pods |
| `resources_per_node` | `{gpu:1, cpu:4}` | Resources per pod |
| `packages_to_install` | `[feast, mlflow, ...]` | Pip packages |

**Volume Mounts:**
- `shared` PVC → `/shared` (model artifacts)
- `feast-config` ConfigMap → Feast remote client config (operator-provided)

**Environment Variables:**
- `MLFLOW_TRACKING_URI` → Internal MLflow service
- `RDZV_TIMEOUT=1800` → DDP rendezvous timeout

In [ ]:
from datetime import datetime
from kubeflow.trainer.options import Name, Labels, PodTemplateOverrides, PodTemplateOverride, PodSpecOverride, ContainerOverride

job_id = datetime.now().strftime('%m%d-%H%M')

# Training function arguments - uses Feast remote client config
func_args = {
    'epochs': EPOCHS,
    'namespace': NAMESPACE,
    'output_dir': '/shared/models',
    'feast_config_path': FEAST_CONFIG_PATH  # Operator-provided Feast remote config
}

job = trainer.train(
    trainer=CustomTrainer(
        func=train_fn,
        func_args=func_args,
        num_nodes=1,
        resources_per_node={'gpu': 1, 'cpu': 4, 'memory': '8Gi'},
        packages_to_install=[
            'feast==0.61.0', 'psycopg2-binary', 'redis',  # Feast remote client (no Ray deps needed)
            'scikit-learn', 'pandas', 'pyarrow', 'joblib',
            'git+https://github.com/red-hat-data-services/mlflow@master'  # RH fork for workspace support
        ],
        env={
            'MLFLOW_TRACKING_URI': MLFLOW_INTERNAL_URI,  # Internal service for pod access
            'MLFLOW_WORKSPACE': MLFLOW_WORKSPACE,
            'MLFLOW_TRACKING_TOKEN': MLFLOW_TOKEN,
            'MLFLOW_TRACKING_INSECURE_TLS': 'true',  # Skip TLS verification for self-signed certs
            'RDZV_TIMEOUT': '1800',
            'NCCL_IB_DISABLE': '1'
        }
    ),
    runtime=runtime,
    options=[
        Name("sales-forecast"),
        Labels({'app': 'sales-forecasting'}),
        PodTemplateOverrides(PodTemplateOverride(
            target_jobs=['node'],
            spec=PodSpecOverride(
                volumes=[
                    {'name': 'shared', 'persistentVolumeClaim': {'claimName': PVC}},
                    {'name': 'feast-config', 'configMap': {'name': 'feast-salesforecasting-client'}},
                    {'name': 'feast-ca', 'configMap': {'name': 'feast-salesforecasting-client-ca', 'items': [{'key': 'service-ca.crt', 'path': 'ca-bundle.crt'}]}}
                ],
                containers=[ContainerOverride(
                    name='node',
                    volume_mounts=[
                        {'name': 'shared', 'mountPath': '/shared'},
                        {'name': 'feast-config', 'mountPath': FEAST_CONFIG_PATH},
                        {'name': 'feast-ca', 'mountPath': '/etc/pki/tls/custom-certs'}
                    ]
                )]
            )
        ))
    ]
)
print(f"✅ Submitted: {job}")

In [ ]:
# trainer.delete_job(job)

## Monitor Training

Wait for job completion (max 1 hour):

In [ ]:
trainer.wait_for_job_status(name=job, status={'Complete', 'Failed'}, timeout=3600)

## View MLflow Results & Model Registry

Check training metrics and registered models:

| Metric | Meaning |
|--------|----------|
| `best_mape` | Mean Absolute Percentage Error (lower is better) |
| `train_loss` | MSE on training set |
| `val_loss` | MSE on validation set |

**Model Registry:** Models are automatically registered as `sales-forecasting-model` with version tags.

In [ ]:
import mlflow
import os
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configure MLflow client with RHOAI auth
if 'MLFLOW_TOKEN' in dir() and MLFLOW_TOKEN:
    os.environ['MLFLOW_TRACKING_TOKEN'] = MLFLOW_TOKEN
os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'
mlflow.set_tracking_uri(MLFLOW_INTERNAL_URI)

# View recent experiment runs
exp_name = f'{MLFLOW_WORKSPACE}/sales-forecasting'
exp = mlflow.get_experiment_by_name(exp_name)
if exp:
    runs = mlflow.search_runs([exp.experiment_id], max_results=5, order_by=['start_time DESC'])
    print("📈 Recent Training Runs:")
    display(runs[['tags.mlflow.runName', 'metrics.best_mape']])
else:
    print(f"Experiment '{exp_name}' not found yet")

# View registered models
print("\n📦 Model Registry:")
try:
    from mlflow.tracking import MlflowClient
    client = MlflowClient()
    for rm in client.search_registered_models(filter_string="name='sales-forecasting-model'"):
        latest = rm.latest_versions[0] if rm.latest_versions else None
        if latest:
            print(f"  Model: {rm.name} | Latest: v{latest.version} | Stage: {latest.current_stage}")
except Exception as e:
    print(f"  No registered models yet: {e}")
    
print(f"\n📊 View in browser: {MLFLOW_EXTERNAL_URI}")

---
## ✅ Training Complete!

**Local Artifacts** (`/shared/models/`):
| File | Purpose |
|------|---------|
| `best_model.pt` | PyTorch model weights |
| `scalers.joblib` | Feature scalers |
| `feature_cols.pkl` | Feature column names |
| `model_metadata.json` | Architecture info + MLflow run ID |

**MLflow Model Registry:**
- Model: `sales-forecasting-model`
- Versioned automatically on each training run
- Tagged with MAPE metrics for comparison
- URI: `runs:/<run_id>/model` (stored in metadata)

**Next:** `03-inference.ipynb` → Deploy model with KServe